# Практика 10. Описательная статистика и аналитические показатели

Сегодня мы не изучаем новых библиотек — только pandas, который уже знаем. Зато учимся
**описывать данные числами** и, главное, понимать, что эти числа означают.

**Данные:** 910 рейсов транспортной компании за февраль–май 2025.

| Часть | О чём | Сколько кода |
|---|---|---|
| 1 | Меры центра: среднее, медиана, мода | 3 строки |
| 2 | Меры разброса: размах, стандартное отклонение, IQR | 3 строки |
| 3 | Меры положения: квартили и перцентили | 2 строки |
| 4 | Частотный анализ: как часто что встречается | 3 строки |
| 5 | Бизнес-метрики и KPI | 5 строк |
| 6 | Интерпретация: что всё это значит | 0 строк, только выводы |

> **Как устроена пара.** Перед каждым заданием идёт готовый разобранный пример: вы его
> запускаете, смотрите результат, а потом делаете то же самое с другим столбцом.
> Кода мало — главное здесь не методы, а умение прочитать число и сказать, что оно значит.

---
## Шаг 0. Проект

```
# терминал
mkdir stats-report
cd stats-report
```

`requirements.txt` — одна строка `pandas`, затем окружение как обычно:

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

---
## Данные

| Колонка | Смысл |
|---|---|
| `trip_id` | номер рейса |
| `date`, `month` | дата и месяц рейса |
| `route` | маршрут |
| `carrier` | перевозчик |
| `cargo_type` | тип груза |
| `client` | клиент |
| `weight_t` | вес груза, тонн |
| `distance_km` | расстояние, км |
| `duration_h` | время в пути, часов |
| `cost_rub` | стоимость рейса, руб. |
| `delay_h` | задержка, часов (0 — рейс вовремя) |

In [1]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт data/trips_q.csv — чистый журнал рейсов за февраль-май 2025.

import csv
import random
from pathlib import Path

random.seed(10)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carriers = ["ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"]
tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}
clients = ["Агрохолдинг Нива", "МеталлПром", "СтройБаза 77",
           "НефтеТрейд", "ЛесЭкспорт", "РитейлСеть"]
days_in_month = {2: 28, 3: 31, 4: 30, 5: 31}

rows = []
trip_no = 1
for month in (2, 3, 4, 5):
    # к лету перевозок становится больше — будет видно на графике динамики
    n_trips = {2: 190, 3: 215, 4: 240, 5: 265}[month]
    for _ in range(n_trips):
        route, distance = random.choice(list(routes.items()))
        cargo = random.choice(list(tariffs))
        carrier = random.choice(carriers)
        weight = round(min(22.0, max(2.0, random.gauss(13, 4.5))), 1)

        # задержки: у большинства рейсов их нет, у остальных — длинный хвост
        if random.random() < 0.72:
            delay = 0
        else:
            delay = random.choice([1, 1, 2, 2, 3, 4, 6, 8, 12, 18])

        speed = random.uniform(46, 64)
        duration = round(distance / speed + delay, 1)
        cost = distance * weight * tariffs[cargo] * random.uniform(0.94, 1.09)

        rows.append({
            "trip_id": f"T{trip_no:04d}",
            "date": f"2025-{month:02d}-{random.randint(1, days_in_month[month]):02d}",
            "month": f"2025-{month:02d}",
            "route": route,
            "carrier": carrier,
            "cargo_type": cargo,
            "client": random.choice(clients),
            "weight_t": weight,
            "distance_km": distance,
            "duration_h": duration,
            "cost_rub": int(round(cost, -1)),
            "delay_h": delay,
        })
        trip_no += 1

Path("data").mkdir(exist_ok=True)
with open("data/trips_q.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Готово. Создан файл data/trips_q.csv")
print("Рейсов:", len(rows))
print("Период: февраль - май 2025")
print("Колонки:", list(rows[0].keys()))

Готово. Создан файл data/trips_q.csv
Рейсов: 910
Период: февраль - май 2025
Колонки: ['trip_id', 'date', 'month', 'route', 'carrier', 'cargo_type', 'client', 'weight_t', 'distance_km', 'duration_h', 'cost_rub', 'delay_h']


In [3]:
%pip install pandas

  Using cached pandas-3.1.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
Using cached pandas-3.1.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas --no-cache-dir



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 15.2 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

df = pd.read_csv("data/trips_q.csv", parse_dates=["date"])

print("рейсов:", len(df))
print("период:", df["date"].min().date(), "-", df["date"].max().date())
df.head()

рейсов: 910
период: 2025-02-01 - 2025-05-31


,trip_id,date,month,route,carrier,cargo_type,client,weight_t,distance_km,duration_h,cost_rub,delay_h
0,T0001,2025-02-17,2025-02,Москва-Казань,ВолгаТранс,Металл,НефтеТрейд,10.3,820,17.2,55370,3
1,T0002,2025-02-20,2025-02,Екатеринбург-Челябинск,РусКарго,Контейнер,СтройБаза 77,11.6,220,8.7,20460,4
2,T0003,2025-02-15,2025-02,Новосибирск-Омск,СибПуть,Металл,МеталлПром,14.6,680,13.2,67130,0
3,T0004,2025-02-10,2025-02,Санкт-Петербург-Петрозаводск,ВолгаТранс,Лес,ЛесЭкспорт,9.9,430,9.3,18800,0
4,T0005,2025-02-17,2025-02,Екатеринбург-Челябинск,СибПуть,Нефтепродукты,МеталлПром,10.5,220,3.8,21010,0


---
# Часть 1. Меры центра

Первый вопрос к любым числовым данным — «а сколько примерно?». На него отвечают три
показателя, и они дают **разные** ответы.

| Показатель | Что это | Метод |
|---|---|---|
| **Среднее** | сумма делится на количество | `.mean()` |
| **Медиана** | значение ровно посередине: половина меньше, половина больше | `.median()` |
| **Мода** | самое часто встречающееся значение | `.mode()` |

Среднее знают все, и именно поэтому его чаще всего применяют неправильно.
Запомните главное: **среднее чувствительно к крупным значениям, медиана — нет.**

### Пример: вес груза

In [4]:
weight = df["weight_t"]

print("средний вес:  ", round(weight.mean(), 1), "т")
print("медианный вес:", round(weight.median(), 1), "т")
print("самый частый: ", weight.mode().iloc[0], "т")

средний вес:   13.1 т
медианный вес: 13.2 т
самый частый:  22.0 т


Здесь все три числа близки — значит вес распределён ровно, без перекоса. Такие данные
действительно можно описать одним числом.

### Задание 1

Посчитайте то же самое для **стоимости рейса** (`cost_rub`) и ответьте на вопросы.

In [15]:
cost = df["cost_rub"]

# 1) среднее, медиана, мода
print("среднее: ", cost.mean().round(2), "руб")
print("медиана: ", cost.median(), "руб")
print("мода:    ", cost.mode().iloc[0], "руб")

# 2) Какое число больше — среднее или медиана? На сколько процентов?
print("среднее больше на ", ((cost.mean() / cost.median()) - 1).round(3) * 100, "%")

# 3) Сколько рейсов дороже среднего? Подсказка: (cost > cost.mean()).sum()
print("дороже среднего:", (df["cost_rub"] > cost.mean()).sum())

# 4) Какую долю от всех рейсов это составляет? Подсказка: вместо .sum() используйте .mean()
print("доля:", ((df["cost_rub"] > cost.mean()).mean().round(2) * 100), "%")

среднее:  44261.95 руб
медиана:  38125.0 руб
мода:     12470 руб
среднее больше на  16.1 %
дороже среднего: 384
доля: 42.0 %


### Разбор: почему среднее больше медианы

Вы должны были получить, что средний рейс стоит **дороже** медианного, и что дороже
среднего — меньше половины рейсов.

Причина: несколько очень дорогих рейсов (22 тонны металла на 820 км) тянут среднее вверх.
Медиана же просто стоит посередине списка и на размер «хвоста» не реагирует.

**Что это значит на практике.** Если сказать руководителю «средний рейс стоит 44 тысячи»,
он решит, что это типичный рейс. А типичный рейс (медианный) стоит 38 тысяч — на 16%
меньше. Такой перекос есть почти во всех денежных данных: в зарплатах, чеках, выручке
по клиентам.

**Правило:** для денег и любых данных с «длинным хвостом» называйте **медиану**,
а среднее — только рядом с ней. Если среднее и медиана близки — можно спокойно
пользоваться средним.

---
# Часть 2. Меры разброса

Центр — это полдела. Два города могут иметь одинаковую среднюю температуру +15,
но в одном круглый год +15, а в другом летом +35 и зимой −5. Нужен второй показатель —
**насколько значения разбросаны**.

| Показатель | Что это | Метод |
|---|---|---|
| **Размах** | максимум минус минимум | `.max() - .min()` |
| **Стандартное отклонение** | насколько в среднем значения отличаются от среднего | `.std()` |
| **IQR** | размах «середины»: Q3 − Q1, то есть диапазон центральных 50% данных | `.quantile(0.75) - .quantile(0.25)` |

Размах считается по двум крайним значениям, поэтому один необычный рейс его полностью
определяет. IQR смотрит только на середину и потому устойчив.

### Пример: время в пути

In [16]:
dur = df["duration_h"]

print("минимум:", dur.min(), "ч | максимум:", dur.max(), "ч")
print("размах: ", round(dur.max() - dur.min(), 1), "ч")
print("стандартное отклонение:", round(dur.std(), 1), "ч")

q1 = dur.quantile(0.25)
q3 = dur.quantile(0.75)
print("IQR:", round(q3 - q1, 1), "ч  (середина данных: от", round(q1, 1), "до", round(q3, 1), "ч)")

минимум: 3.4 ч | максимум: 32.6 ч
размах:  29.2 ч
стандартное отклонение: 4.9 ч
IQR: 6.5 ч  (середина данных: от 7.2 до 13.7 ч)


### Задание 2

Посчитайте разброс **стоимости рейса**.

1. Минимум, максимум и размах.
2. Стандартное отклонение.
3. Q1, Q3 и IQR. В каком диапазоне стоимости лежит «середина» рейсов?
4. Посчитайте **коэффициент вариации**: стандартное отклонение, делённое на среднее,
   в процентах. Если он больше 30% — данные считаются сильно разбросанными,
   и одним средним их описывать нельзя. Что получилось у вас?

In [19]:
# 1)
print("min:", cost.min(), "| max:", cost.max(), "| размах:", cost.max() - cost.min())

# 2)
print("стандартное отклонение:", cost.std())

# 3)
q1 = cost.quantile(0.25)
q3 = cost.quantile(0.75)
print("середина данных: от", q1, "до", q3, "| IQR:", q3 - q1)

# 4)
print("коэффициент вариации:", round((cost.std() / cost.mean()) * 100, 2), "%")

min: 3170 | max: 175010 | размах: 171840
стандартное отклонение: 26168.727279172734
середина данных: от 24580.0 до 58410.0 | IQR: 33830.0
коэффициент вариации: 59.12 %


> данные сильно разбросаны

---
# Часть 3. Меры положения

**Перцентиль** — значение, ниже которого лежит заданная доля данных.
90-й перцентиль стоимости = 81 500 означает: 90% рейсов дешевле этой суммы,
а 10% — дороже.

Квартили — это просто перцентили 25, 50 и 75.

Удобнее всего смотреть всё сразу методом `.describe()`.

In [20]:
# Отдельные перцентили
print("10% самых дешёвых рейсов — дешевле", round(df["cost_rub"].quantile(0.10)), "руб.")
print("10% самых дорогих рейсов — дороже ", round(df["cost_rub"].quantile(0.90)), "руб.")
print()

# Всё сразу по нескольким столбцам
df[["cost_rub", "weight_t", "distance_km", "delay_h"]].describe().round(1)

10% самых дешёвых рейсов — дешевле 15997 руб.
10% самых дорогих рейсов — дороже  81557 руб.



,cost_rub,weight_t,distance_km,delay_h
count,910.0,910.0,910.0,910.0
mean,44261.9,13.1,514.2,1.3
std,26168.7,4.4,190.2,3.3
min,3170.0,2.0,220.0,0.0
25%,24580.0,10.1,360.0,0.0
50%,38125.0,13.2,430.0,0.0
75%,58410.0,16.1,680.0,1.0
max,175010.0,22.0,820.0,18.0


### Задание 3

1. Посмотрите на столбец `delay_h` в таблице `describe()` выше. Чему равна медиана
   (строка `50%`)? А среднее? Как такое может быть?
2. Посчитайте, какую **долю выручки** приносят 10% самых дорогих рейсов.
   Подсказка по шагам:
   ```python
   граница = df["cost_rub"].quantile(0.90)
   топ = df[df["cost_rub"] >= граница]
   доля = топ["cost_rub"].sum() / df["cost_rub"].sum() * 100
   ```
3. Сколько рейсов попало в этот топ-10%?

In [27]:

# 2)
quantile_90 = df["cost_rub"].quantile(0.90)
top = df[df["cost_rub"] >= quantile_90]
print((top["cost_rub"].sum() / df["cost_rub"].sum() * 100).round(2), "%")

# 3)
print(top.shape[0])

22.49 %
91


> **Ответ на вопрос 1:** медиана задержки равна 0, а среднее 1.3,
> такое может быть, если много рейсов без задержек, а те, у которых она есть - она большая

---
# Часть 4. Частотный анализ

Для текстовых столбцов среднее посчитать нельзя — там считают **частоты**: сколько раз
каждое значение встречается.

| Задача | Метод |
|---|---|
| сколько раз встречается каждое значение | `.value_counts()` |
| то же в долях | `.value_counts(normalize=True)` |
| разбить числа на интервалы | `pd.cut(...)` |

Числовые данные тоже можно разложить по частотам — для этого их сначала разбивают
на интервалы (мы это делали на практике 6).

### Пример: типы груза

In [28]:
print("сколько рейсов с каждым грузом:")
print(df["cargo_type"].value_counts())
print()

print("то же в процентах:")
print((df["cargo_type"].value_counts(normalize=True) * 100).round(1))

сколько рейсов с каждым грузом:
cargo_type
Зерно             164
Металл            163
Стройматериалы    160
Лес               156
Контейнер         139
Нефтепродукты     128
Name: count, dtype: int64

то же в процентах:
cargo_type
Зерно             18.0
Металл            17.9
Стройматериалы    17.6
Лес               17.1
Контейнер         15.3
Нефтепродукты     14.1
Name: proportion, dtype: float64


In [29]:
# Числовой столбец: раскладываем стоимость по интервалам
df["cost_group"] = pd.cut(
    df["cost_rub"],
    bins=[0, 20000, 50000, 100000, 1000000],
    labels=["до 20 тыс", "20-50 тыс", "50-100 тыс", "более 100 тыс"],
)

freq = df["cost_group"].value_counts().sort_index().to_frame("рейсов")
freq["доля рейсов, %"] = (freq["рейсов"] / len(df) * 100).round(1)
freq

,рейсов,"доля рейсов, %"
cost_group,,
до 20 тыс,157,17.3
20-50 тыс,431,47.4
50-100 тыс,287,31.5
более 100 тыс,35,3.8


Такая таблица называется **таблицей распределения**. Она отвечает на вопрос
«каких рейсов много, а каких мало» лучше любого среднего.

### Задание 4

1. Постройте частотную таблицу по перевозчикам (`carrier`) — в количестве и в процентах.
   Распределение ровное или кто-то выделяется?
2. Добавьте в таблицу `freq` из примера выше столбец с **выручкой** по каждой группе.
   Подсказка: `df.groupby("cost_group", observed=True)["cost_rub"].sum()`
3. Посчитайте долю выручки каждой группы в процентах.
4. Сравните две доли: группа «до 20 тыс» — какую долю **рейсов** она составляет
   и какую долю **выручки** приносит? Что это говорит о структуре бизнеса?

In [38]:
# 1)
display(df["carrier"].value_counts().reset_index())

# 2)
freq["выручка"] = df.groupby("cost_group", observed=True)["cost_rub"].sum()

# 3)
freq["доля выручки, %"] = (freq["выручка"] / freq["выручка"].sum() * 100).round(2)

# 4) ответ словами
display(freq)

,carrier,count
0,АвтоДорСервис,201
1,СибПуть,182
2,ТрансЛогистик,182
3,РусКарго,173
4,ВолгаТранс,172


,рейсов,"доля рейсов, %",выручка,"доля выручки, %"
cost_group,,,,
до 20 тыс,157,17.3,2230710,5.54
20-50 тыс,431,47.4,14525960,36.06
50-100 тыс,287,31.5,19467230,48.33
более 100 тыс,35,3.8,4054470,10.07


4) > она составляет 17 % рейсов, и 5 % выручки, т.е. для бизнеса она не является целевой категорией, как доставки за 20-50 и 50-100 тыс

---
# Часть 5. Бизнес-метрики и KPI

**KPI** (key performance indicator) — ключевой показатель, по которому судят о работе.
От обычной статистики он отличается тем, что его смысл задаёт бизнес, а не математика.

Пять KPI транспортной компании:

| KPI | Как считается | Что показывает |
|---|---|---|
| Выручка | сумма `cost_rub` | сколько заработали |
| Средний чек | средняя стоимость рейса | масштаб типичной сделки |
| Выручка на километр | выручка ÷ сумма расстояний | эффективность плеча |
| Выручка на тонно-километр | выручка ÷ сумма (вес × расстояние) | цена работы без влияния размера груза |
| **OTD** (on-time delivery) | доля рейсов без задержки | качество исполнения |

OTD — главный KPI в логистике. Его считают как долю рейсов, пришедших вовремя:
`(df["delay_h"] == 0).mean() * 100`.

### Пример: KPI по всей компании

In [39]:
revenue = df["cost_rub"].sum()

print("ВЫРУЧКА И ОБЪЁМ")
print(f"  выручка:        {revenue:>12,} руб.".replace(",", " "))
print(f"  рейсов:         {len(df):>12}")
print(f"  средний чек:    {df['cost_rub'].mean():>12,.0f} руб.".replace(",", " "))
print()
print("ЭФФЕКТИВНОСТЬ")
print(f"  выручка на км:         {revenue / df['distance_km'].sum():>8.1f} руб.")
print(f"  выручка на тонно-км:   {revenue / (df['weight_t'] * df['distance_km']).sum():>8.2f} руб.")
print(f"  средняя загрузка:      {df['weight_t'].mean():>8.1f} т")
print()
print("КАЧЕСТВО")
print(f"  OTD (рейсов вовремя):  {(df['delay_h'] == 0).mean() * 100:>8.1f} %")
print(f"  средняя задержка:      {df['delay_h'].mean():>8.2f} ч")

ВЫРУЧКА И ОБЪЁМ
  выручка:          40 278 370 руб.
  рейсов:                  910
  средний чек:          44 262 руб.

ЭФФЕКТИВНОСТЬ
  выручка на км:             86.1 руб.
  выручка на тонно-км:       6.55 руб.
  средняя загрузка:          13.1 т

КАЧЕСТВО
  OTD (рейсов вовремя):      74.1 %
  средняя задержка:          1.33 ч


### Задание 5

Посчитайте KPI **по каждому перевозчику** и выясните, кто работает лучше.

Используйте `groupby` с именованной агрегацией — как на практике 9:

```python
kpi = df.groupby("carrier").agg(
    trips=("trip_id", "count"),
    revenue=("cost_rub", "sum"),
    avg_ticket=("cost_rub", "mean"),
    otd=("delay_h", lambda s: (s == 0).mean() * 100),
).round(1)
```

1. Посчитайте эту таблицу и отсортируйте по выручке.
2. Кто лидер по выручке? А по OTD? Это один и тот же перевозчик?
3. Добавьте столбец `rev_per_km` — выручка на километр по каждому перевозчику.
   Подсказка: посчитайте две группировки и поделите одну на другую.
4. Кому бы вы отдали следующий срочный заказ и почему? А самый выгодный?

In [44]:
# 1)
kpi = df.groupby("carrier").agg(
    trips=("trip_id", "count"),
    revenue=("cost_rub", "sum"),
    avg_ticket=("cost_rub", "mean"),
    otd=("delay_h", lambda s: (s == 0).mean() * 100),
).round(1).sort_values("revenue", ascending=False)
print(kpi)

# 2) ответ словами

# 3)
kpi["rev_per_km"] = df.groupby("carrier")["cost_rub"].sum() / df.groupby("carrier")["distance_km"].sum()
display(kpi)
# 4) ответ словами

               trips  revenue  avg_ticket   otd
carrier                                        
АвтоДорСервис    201  8719550     43380.8  70.1
ТрансЛогистик    182  8370650     45992.6  75.3
СибПуть          182  7864100     43209.3  74.7
РусКарго         173  7791010     45034.7  72.8
ВолгаТранс       172  7533060     43796.9  77.9


,trips,revenue,avg_ticket,otd,rev_per_km
carrier,,,,,
АвтоДорСервис,201,8719550,43380.8,70.1,82.767442
ТрансЛогистик,182,8370650,45992.6,75.3,89.106344
СибПуть,182,7864100,43209.3,74.7,82.458845
РусКарго,173,7791010,45034.7,72.8,87.470641
ВолгаТранс,172,7533060,43796.9,77.9,89.476897


2) > по выручке главный - АвтоДорСервис, а по доле рейсов без задержки - ВолгаТранс
3) > самый срочный - Волгатранс за низкий OTD, а самый выгодный у СибПуть - низкая цена за километр

---
# Часть 6. Интерпретация: что всё это значит

Самая важная часть. Числа посчитаны — теперь нужно сказать, что они означают для бизнеса.
Таблица ниже готова, считать ничего не нужно: читайте и делайте выводы.

In [45]:
by_month = df.groupby("month").agg(
    trips=("trip_id", "count"),
    revenue=("cost_rub", "sum"),
    avg_ticket=("cost_rub", "mean"),
    avg_weight=("weight_t", "mean"),
    avg_distance=("distance_km", "mean"),
    otd=("delay_h", lambda s: (s == 0).mean() * 100),
).round(1)

by_month["рост выручки, %"] = (by_month["revenue"].pct_change() * 100).round(1)
by_month

,trips,revenue,avg_ticket,avg_weight,avg_distance,otd,"рост выручки, %"
month,,,,,,,
2025-02,190,8491320,44691.2,12.5,536.6,75.3,NaN
2025-03,215,9490750,44143.0,13.2,522.1,75.3,11.8
2025-04,240,11035900,45982.9,13.6,504.3,72.5,16.3
2025-05,265,11260400,42492.1,13.0,500.6,73.6,2.0


### Задание 6. Выводы

Ответьте на вопросы по таблице выше. Код не нужен — нужны числа из таблицы и объяснение.

1. Сколько рейсов было в феврале и сколько в мае? На сколько процентов выросло количество?
2. На сколько процентов выросла выручка за тот же период?
3. **Главный вопрос.** В мае рейсов больше всего, а рост выручки к апрелю — всего 2%.
   Посмотрите на столбцы `avg_ticket`, `avg_weight` и `avg_distance`.
   Чем объясняется такое расхождение?
4. Что произошло с OTD за четыре месяца? Это связано с ростом числа рейсов?
5. Напишите три предложения для руководителя: что происходит с бизнесом и на что
   обратить внимание.

> **Как отвечать.** Плохой ответ: «выручка выросла на 32.6%». Хороший: «выручка выросла
> на 32.6%, но за счёт количества рейсов, а не их стоимости: средний чек даже снизился.
> Значит компания берёт больше более мелких заказов».

> **Мои ответы:**
>
> 1. февраль - 190, май - 265, на 40%
> 2. на 30.1%, но при этом все остальные метрики снизились - увеличилась выручка за счёт количества, а не качества
> 3. вес увеличился, средний чек уменьшился, расстояния уменьшились, т.е. стало больше мелких тяжелых перевозок, видимо, необходимо увеличить влияние веса на перевозку
> 4. За счёт увеличения количества заказов снизилась метрика OTD, видимо, компаниям стало тяжелее следить за поставками
>
> **Для руководителя:**
>
> 1. попробовать увеличить влияние веса на итоговую цену перевозки, т.к. выручка плохо увеличивается, т.к. все метрики уменбшаются кроме веса
> 2. обратить внимание на сектор с низкой ценой - он занимает малую часть перевозок, возможно, требуется упрощение процесса перевозок для мелких клиентов
> 3. обратить внимание на контроль OTD - из-за наплыва новых пользователей сложнее стало следить ха этой метрикой

---
## Шпаргалка

```python
# центр
df["col"].mean()        # среднее
df["col"].median()      # медиана
df["col"].mode()        # мода

# разброс
df["col"].max() - df["col"].min()     # размах
df["col"].std()                       # стандартное отклонение
df["col"].quantile(0.75) - df["col"].quantile(0.25)   # IQR

# положение
df["col"].quantile(0.9)               # 90-й перцентиль
df["col"].describe()                  # всё сразу

# частоты
df["col"].value_counts()
df["col"].value_counts(normalize=True)
pd.cut(df["col"], bins=[...], labels=[...])

# KPI
(df["delay_h"] == 0).mean() * 100     # доля в процентах
```

## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 10: описательная статистика и KPI"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: три меры центра, объяснено, почему среднее больше медианы
- [ ] Задание 2: размах, стандартное отклонение, IQR, коэффициент вариации
- [ ] Задание 3: посчитана доля выручки топ-10% рейсов, объяснена медиана задержки
- [ ] Задание 4: частотные таблицы, сравнение доли рейсов и доли выручки
- [ ] Задание 5: KPI по перевозчикам, выбран исполнитель с обоснованием
- [ ] Задание 6: письменные выводы с числами
- [ ] Всё запушено в репозиторий

## Что унести с пары

Два правила, которые работают всегда:

1. **Одно число не описывает данные.** Нужны как минимум два: центр и разброс.
   «Средний рейс 44 тысячи» — не ответ, пока не сказано, что половина рейсов
   стоит от 25 до 58 тысяч.
2. **Для денег смотрите медиану.** Среднее у денежных данных почти всегда завышено
   из-за крупных значений: у нас дороже среднего оказалось лишь 42% рейсов.

И главное: посчитать показатель — работа на минуту, объяснить его — работа аналитика.
Числа сами ничего не значат, значение им придаёт контекст предметной области.